# Criação de uma automação usando PlayWright

In [1]:
#from playwright.sync_api import sync_playwright
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright

In [3]:
import pandas as pd
import seaborn as sns

In [2]:
import subprocess
import sys
import os
import json
import pandas as pd

# 1. SCRIPT PARA O GOOGLE
script_content = """
from playwright.sync_api import sync_playwright
import json
import time

def run():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        context = browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")
        page = context.new_page()
        
        try:
            print("Abrindo Google...")
            page.goto("https://www.google.com.br")
            
            # Aceita cookies se aparecer (comum em novas instâncias)
            try:
                page.click('button:has-text("Aceitar tudo")', timeout=3000)
            except:
                pass

            # Digita a busca
            search_box = page.wait_for_selector('textarea[name="q"]', timeout=10000)
            search_box.fill('pedal de guitarra preço')
            search_box.press('Enter')
            
            # Espera os resultados de "Shopping" ou resultados orgânicos
            page.wait_for_selector('#search', timeout=10000)
            time.sleep(2)

            # Extrai os títulos e links dos resultados orgânicos principais
            results = page.query_selector_all('div.g')
            dados = []

            for res in results[:8]: # Pega os 8 primeiros
                titulo_elem = res.query_selector('h3')
                link_elem = res.query_selector('a')
                
                if titulo_elem and link_elem:
                    dados.append({
                        "Título": titulo_elem.inner_text(),
                        "Link": link_elem.get_attribute('href')
                    })
            
            with open("dados_google.json", "w", encoding="utf-8") as f:
                json.dump(dados, f, ensure_ascii=False)
                
            print(f"Sucesso! {len(dados)} links encontrados no Google.")

        except Exception as e:
            print(f"Erro no Google: {e}")
        finally:
            browser.close()

if __name__ == "__main__":
    run()
"""

# 2. SALVA E EXECUTA
with open("scraper_google.py", "w", encoding="utf-8") as f:
    f.write(script_content)

print("--- PESQUISANDO NO GOOGLE ---")
subprocess.run([sys.executable, "scraper_google.py"])

# 3. EXIBE RESULTADO
if os.path.exists("dados_google.json"):
    df = pd.read_json("dados_google.json")
    display(df)
else:
    print("Ocorreu um erro na pesquisa.")


--- PESQUISANDO NO GOOGLE ---
Ocorreu um erro na pesquisa.


In [ ]:
dados_json

Daqui para baixo são outras versões

In [ ]:
def pesquisar_pedal_mercado_livre():
    with sync_playwright() as p:
        # Abre o navegador (headless=False permite ver o navegador abrindo)
        browser = p.chromium.launch(headless=False) 
        page = browser.new_page()
        
        # Acessa o site do Mercado Livre
        page.goto("https://www.mercadolivre.com.br")
        
        # Faz a pesquisa
        search_bar = page.locator("input[name='as_word']")
        search_bar.fill("pedal de guitarra")
        search_bar.press("Enter")
        
        # Aguarda os resultados carregarem
        page.wait_for_selector(".ui-search-results")
        
        # Captura o primeiro produto da lista
        primeiro_produto = page.locator(".ui-search-result__wrapper").first
        
        # Extrai nome e preço
        nome = primeiro_produto.locator(".ui-search-item__title").inner_text()
        preco = primeiro_produto.locator(".andes-money-amount__fraction").first.inner_text()
        
        print(f"Produto: {nome}")
        print(f"Preço: R$ {preco}")
        
        # Fecha o navegador
        browser.close()

if __name__ == "__main__":
    pesquisar_pedal_mercado_livre()

In [ ]:
with async_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto("https://exemplo.com/produtos") # substitua pela URL real

    # Coletar os dados (supondo que os produtos estão em elementos com classe 'produto')
    produtos = page.query_selector_all('.produto')
    
    dados = []
    for produto in produtos:
        nome = produto.query_selector('.nome').inner_text()
        preco = produto.query_selector('.preco').inner_text()
        dados.append({'Nome': nome, 'Preço': preco})
    
    # Criar um DataFrame a partir da lista de dados
    df = pd.DataFrame(dados)
    
    # Opcional: salvar em um arquivo CSV
    df.to_csv('produtos.csv', index=False)
    
    browser.close()